# Capstone — Refresh Opportunity Scoring

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanraza04/flyrank_intern_content/blob/capstone-refresh/work/notebooks/capstone.ipynb?flush_cache=true)

This is the capstone notebook. It builds an honest content-review ranking from the FlyRank warehouse and will mirror the public research paper.

## Question

Which established content items should an editor review first because their search impressions are likely to fall in the following 28 days? One row is one client-content item at a decision date. The output is a ranked review queue. A wrong call costs editorial attention, so this is decision support, not automated publishing.

In [ ]:
%pip -q install duckdb pandas pyarrow scikit-learn

import os
from pathlib import Path

if not Path('work/scripts/capstone_data.py').exists():
    !git clone -q --branch capstone-refresh https://github.com/hassanraza04/flyrank_intern_content.git
    %cd flyrank_intern_content

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Set HF_TOKEN in Colab Secrets. Do not paste a token into this notebook.')

## Data and methodology

The model uses `fact_content_daily_performance` through DuckDB over `hf://`. It excludes GA4 engagement fields because availability is uneven, plus IDs, query text, domains, URLs, and all future-window values. Features summarize two completed 28-day windows. The label is 1 when the following 28-day impressions are below 80% of the current window. June 2026 is the sealed final outcome period.

In [ ]:
import json
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

from work.scripts.capstone_data import build_feature_frame, create_connection
from work.scripts.capstone_utils import FEATURE_COLUMNS, add_baseline_score, evaluate_ranking, validate_feature_columns

cache = Path('work/outputs/capstone_features.parquet')
features = pd.read_parquet(cache) if cache.exists() else build_feature_frame(create_connection(HF_TOKEN), cache)
validate_feature_columns(FEATURE_COLUMNS)

train = features[features.cohort_id.isin(['2025-12','2026-01','2026-02','2026-03','2026-04'])]
validation = features[features.cohort_id == '2026-05']
sealed = features[features.cohort_id == '2026-06-sealed']
print({'train_rows': len(train), 'validation_rows': len(validation), 'sealed_rows': len(sealed)})

## Results versus baseline

The transparent baseline prioritizes visible pages whose recent impressions have already fallen, with a small extra weight for worsening average position. It is compared with logistic regression and a shallow gradient-boosting model on the same time-aware validation cohort. The preferred method is frozen before the sealed June outcome cohort is evaluated.

In [ ]:
models = {
    'logistic_regression': make_pipeline(SimpleImputer(strategy='median'), LogisticRegression(max_iter=1000, C=0.5, random_state=42)),
    'hist_gradient_boosting': make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.08, max_iter=150, random_state=42)),
}
rows = []
for name, model in models.items():
    model.fit(train[FEATURE_COLUMNS], train.is_declining_proxy)
    rows.append({'method': name, **evaluate_ranking(validation.is_declining_proxy, model.predict_proba(validation[FEATURE_COLUMNS])[:, 1], 100)})
validation_baseline = add_baseline_score(validation)
rows.append({'method': 'momentum_baseline', **evaluate_ranking(validation_baseline.is_declining_proxy, validation_baseline.baseline_score, 100)})
comparison = pd.DataFrame(rows).sort_values('precision_at_k', ascending=False).reset_index(drop=True)
comparison

In [ ]:
selected_name = comparison.loc[0, 'method']
all_development = pd.concat([train, validation])
selected_model = None if selected_name == 'momentum_baseline' else models[selected_name].fit(all_development[FEATURE_COLUMNS], all_development.is_declining_proxy)
sealed_baseline = add_baseline_score(sealed)
baseline_metrics = evaluate_ranking(sealed.is_declining_proxy, sealed_baseline.baseline_score, 100)
selected_scores = sealed_baseline.baseline_score if selected_model is None else selected_model.predict_proba(sealed[FEATURE_COLUMNS])[:, 1]
model_metrics = evaluate_ranking(sealed.is_declining_proxy, selected_scores, 100)
metrics = {'cohort':'2026-06-sealed','selected_method':selected_name,'baseline':baseline_metrics,'model':model_metrics,'base_rate':model_metrics['base_rate'],'feature_columns':FEATURE_COLUMNS,'seed':42}
print(json.dumps(metrics, indent=2))

## Limitations and ranked recommendations

This is a measured, directional proxy, not proof that refreshing a page causes a visibility gain. Search demand, seasonality, tracking changes, and indexing context can all affect impressions. The recommended action is human review: refresh review for high-risk visible pages, search-intent review when average position worsens, snippet review when CTR is low, and monitoring when the signal is weak.

## Reproducibility and credit

Run this notebook from a fresh Colab runtime with a secret-based `HF_TOKEN`. The code uses fixed random seed 42 and commits only code, aggregates, and public-safe narrative. Built on the [FlyRank ML Internship dataset](https://flyrank.ai).